In [5]:
import os
import numpy as np
import soundfile as sf
import librosa
from pathlib import Path
from IPython.display import Audio

In [2]:
# ================================
# 설정
# ================================
INPUT_DIR = "/home/isol/work/data/train/wakeword_clean"         # 정제된 원본 wakeword wav 폴더
OUTPUT_DIR = "/home/isol/work/data/train/wakeword_augmented"    # 증강 결과 저장 폴더
SAMPLE_RATE = 16000
TARGET_LEN = SAMPLE_RATE                                        # 1초 = 16000 샘플

os.makedirs(OUTPUT_DIR, exist_ok=True)

In [3]:
# ================================
# 유틸
# ================================
def load_audio(path):
    """wav 로드 후 모노 16kHz로 리샘플"""
    audio, sr = librosa.load(path, sr=SAMPLE_RATE, mono=True)
    return audio

def save_audio(audio, path):
    sf.write(path, audio, SAMPLE_RATE)

In [7]:
# ================================
# 증강 함수들
# ================================
def augment_pitch_shift(audio, sr, n_steps):
    """피치 조절 (semitone 단위)"""
    return librosa.effects.pitch_shift(audio, sr=sr, n_steps=n_steps)

def fix_length(audio, target_len=TARGET_LEN):
    """1초 길이로 맞추기 (pad or crop)"""
    if len(audio) < target_len:
        # 중앙 정렬로 패딩
        pad_total = target_len - len(audio)
        pad_left  = pad_total // 2
        pad_right = pad_total - pad_left
        audio = np.pad(audio, (pad_left, pad_right), mode='constant')
    else:
        # 뒷 부분 자름
        audio = audio[:target_len]
    return audio

def augment_time_stretch(audio, rate):
    """
    속도 조절 (피치 유지)
    rate > 1.0 : 빠르게 / rate < 1.0 : 느리게
    """
    stretched = librosa.effects.time_stretch(audio, rate=rate)
    return fix_length(stretched)

def augment_volume_scale(audio, gain):
    """음량 조절"""
    scaled = audio * gain
    # 클리핑 방지
    if np.max(np.abs(scaled)) > 1.0:
        scaled = scaled / np.max(np.abs(scaled)) * 0.95
    return scaled

def augment_add_gaussian_noise(audio, noise_level=0.005):
    """미세한 백색 소음 추가"""
    noise = np.random.randn(len(audio)) * noise_level
    return np.clip(audio + noise, -1.0, 1.0)

In [21]:
audio, sr = librosa.load("/home/isol/work/data/train/wakeword_clean/20251204_131913_MIC0.wav", sr=16000)
Audio(audio, rate=sr)

In [24]:
pitch_audio = augment_pitch_shift(audio, sr, +0.5)
Audio(pitch_audio, rate=sr)

In [29]:
stretch_audio = augment_time_stretch(audio, 0.95)
Audio(stretch_audio, rate=sr)